In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Creates a .db file in sql folder
engine = create_engine('sqlite:///../sql/f1_analytics.db')
print("Database created!")

Database created!


In [2]:
import os
# List all race names
races = ['bahrain', 'saudi_arabia', 'japan', 'miami', 'monaco']

# Load and combine all results files into one big table
all_results = []

for race in races:
    file = f'../data/{race}_2025_results.csv'
    if os.path.exists(file):
        df = pd.read_csv(file)
        all_results.append(df)
    else:
        print(f"Missing: {file}")

results_combined = pd.concat(all_results, ignore_index=True)
print(f"Total rows: {len(results_combined)}")
print(results_combined.head())

Total rows: 100
   DriverNumber Abbreviation         FullName  TeamName  Position  Points  \
0            81          PIA    Oscar Piastri   McLaren       1.0    25.0   
1            63          RUS   George Russell  Mercedes       2.0    18.0   
2             4          NOR     Lando Norris   McLaren       3.0    15.0   
3            16          LEC  Charles Leclerc   Ferrari       4.0    12.0   
4            44          HAM   Lewis Hamilton   Ferrari       5.0    10.0   

   GridPosition    Status  Season     Race  
0           1.0  Finished    2025  Bahrain  
1           3.0  Finished    2025  Bahrain  
2           6.0  Finished    2025  Bahrain  
3           2.0  Finished    2025  Bahrain  
4           9.0  Finished    2025  Bahrain  


In [3]:
all_laps = []

for race in races:
    file = f'../data/{race}_2025_laps.csv'
    if os.path.exists(file):
        df = pd.read_csv(file)
        all_laps.append(df)
    else:
        print(f"Missing: {file}")

laps_combined = pd.concat(all_laps, ignore_index=True)
print(f"Total rows: {len(laps_combined)}")
print(laps_combined.head())

Total rows: 5515
  Driver  LapNumber                 LapTime             Sector1Time  \
0    PIA        1.0  0 days 00:01:38.693000                     NaN   
1    PIA        2.0  0 days 00:01:37.492000  0 days 00:00:31.139000   
2    PIA        3.0  0 days 00:01:38.083000  0 days 00:00:31.306000   
3    PIA        4.0  0 days 00:01:38.133000  0 days 00:00:31.326000   
4    PIA        5.0  0 days 00:01:38.043000  0 days 00:00:31.305000   

              Sector2Time             Sector3Time Compound  TyreLife  \
0  0 days 00:00:42.130000  0 days 00:00:23.956000     SOFT       4.0   
1  0 days 00:00:42.343000  0 days 00:00:24.010000     SOFT       5.0   
2  0 days 00:00:42.727000  0 days 00:00:24.050000     SOFT       6.0   
3  0 days 00:00:42.796000  0 days 00:00:24.011000     SOFT       7.0   
4  0 days 00:00:42.690000  0 days 00:00:24.048000     SOFT       8.0   

  IsPersonalBest  LapTime_sec  Sector1_sec  Sector2_sec  Sector3_sec  Season  \
0          False       98.693          NaN 

In [4]:
all_weather = []

for race in races:
    file = f'../data/{race}_2025_weather.csv'
    if os.path.exists(file):
        df = pd.read_csv(file)
        all_weather.append(df)
    else:
        print(f"Missing: {file}")

weather_combined = pd.concat(all_weather, ignore_index=True)
print(f"Total rows: {len(weather_combined)}")
print(weather_combined.head())

Total rows: 749
                     Time  AirTemp  TrackTemp  Humidity  WindSpeed  Rainfall  \
0  0 days 00:00:14.542000     27.9       34.5      45.0        2.9     False   
1  0 days 00:01:14.570000     27.8       34.4      45.0        2.9     False   
2  0 days 00:02:14.574000     27.9       34.6      45.0        2.7     False   
3  0 days 00:03:14.590000     27.9       34.4      45.0        3.2     False   
4  0 days 00:04:14.608000     27.8       34.4      45.0        2.9     False   

   Season     Race  
0    2025  Bahrain  
1    2025  Bahrain  
2    2025  Bahrain  
3    2025  Bahrain  
4    2025  Bahrain  


In [5]:
# if_exists='replace' means it overwrites the table if i run this again
results_combined.to_sql('race_results', engine, if_exists='replace', index=False)
laps_combined.to_sql('lap_details', engine, if_exists='replace', index=False)
weather_combined.to_sql('weather', engine, if_exists='replace', index=False)

print("All tables written to database!")

All tables written to database!


In [6]:
# Reading directly from SQL to confirm the data is really there
test = pd.read_sql("SELECT * FROM race_results LIMIT 10", engine)
print(test)

   DriverNumber Abbreviation         FullName         TeamName  Position  \
0            81          PIA    Oscar Piastri          McLaren       1.0   
1            63          RUS   George Russell         Mercedes       2.0   
2             4          NOR     Lando Norris          McLaren       3.0   
3            16          LEC  Charles Leclerc          Ferrari       4.0   
4            44          HAM   Lewis Hamilton          Ferrari       5.0   
5             1          VER   Max Verstappen  Red Bull Racing       6.0   
6            10          GAS     Pierre Gasly           Alpine       7.0   
7            31          OCO     Esteban Ocon     Haas F1 Team       8.0   
8            22          TSU     Yuki Tsunoda  Red Bull Racing       9.0   
9            87          BEA   Oliver Bearman     Haas F1 Team      10.0   

   Points  GridPosition    Status  Season     Race  
0    25.0           1.0  Finished    2025  Bahrain  
1    18.0           3.0  Finished    2025  Bahrain  
2   

In [7]:
query = """
SELECT Race, FullName, TeamName, Position
FROM race_results
WHERE Position <=3
ORDER BY Race, Position
"""
podiums = pd.read_sql(query, engine)
print(podiums)

            Race         FullName         TeamName  Position
0        Bahrain    Oscar Piastri          McLaren       1.0
1        Bahrain   George Russell         Mercedes       2.0
2        Bahrain     Lando Norris          McLaren       3.0
3          Japan   Max Verstappen  Red Bull Racing       1.0
4          Japan     Lando Norris          McLaren       2.0
5          Japan    Oscar Piastri          McLaren       3.0
6          Miami    Oscar Piastri          McLaren       1.0
7          Miami     Lando Norris          McLaren       2.0
8          Miami   George Russell         Mercedes       3.0
9         Monaco     Lando Norris          McLaren       1.0
10        Monaco  Charles Leclerc          Ferrari       2.0
11        Monaco    Oscar Piastri          McLaren       3.0
12  Saudi Arabia    Oscar Piastri          McLaren       1.0
13  Saudi Arabia   Max Verstappen  Red Bull Racing       2.0
14  Saudi Arabia  Charles Leclerc          Ferrari       3.0


In [9]:
query = """
SELECT FullName, TeamName, SUM(Points) as TotalPoints
FROM race_results
GROUP BY FullName, TeamName
ORDER BY TotalPoints DESC
"""
standings = pd.read_sql(query, engine)
print(standings)

             FullName         TeamName  TotalPoints
0       Oscar Piastri          McLaren        105.0
1        Lando Norris          McLaren         88.0
2      Max Verstappen  Red Bull Racing         75.0
3     Charles Leclerc          Ferrari         63.0
4      George Russell         Mercedes         53.0
5      Lewis Hamilton          Ferrari         36.0
6      Kimi Antonelli         Mercedes         24.0
7     Alexander Albon         Williams         16.0
8        Isack Hadjar     Racing Bulls         13.0
9        Esteban Ocon     Haas F1 Team         10.0
10       Carlos Sainz         Williams          7.0
11       Pierre Gasly           Alpine          6.0
12        Liam Lawson     Racing Bulls          4.0
13       Yuki Tsunoda  Red Bull Racing          3.0
14     Oliver Bearman     Haas F1 Team          2.0
15    Fernando Alonso     Aston Martin          0.0
16   Franco Colapinto           Alpine          0.0
17  Gabriel Bortoleto      Kick Sauber          0.0
18        Ja

In [10]:
query = """
SELECT Driver, ROUND(AVG(LapTime_sec), 3) as AvgLapTime
FROM lap_details
WHERE Race = 'Bahrain'
AND LapTime_sec IS NOT NULL
GROUP BY Driver
ORDER BY AvgLapTime ASC
"""
avg_laps = pd.read_sql(query, engine)
print(avg_laps)

   Driver  AvgLapTime
0     PIA      99.524
1     RUS      99.949
2     NOR      99.984
3     LEC     100.019
4     HAM     100.272
5     TSU     100.589
6     ANT     100.647
7     LAW     100.656
8     ALB     100.663
9     ALO     100.858
10    STR     100.934
11    BOR     100.972
12    VER     101.295
13    GAS     101.323
14    OCO     101.468
15    BEA     101.527
16    DOO     101.618
17    HUL     101.630
18    HAD     101.680
19    SAI     102.383
